In [0]:
from pyspark.sql.functions import col, when

In [0]:
quality_events = spark.table(
    "workspace.pharma_silver.silver_quality_events"
)

batches = spark.table(
    "workspace.pharma_silver.silver_batches"
)

drugs = spark.table(
    "workspace.pharma_silver.silver_drugs"
)

In [0]:
df_quality_gold = (
    quality_events
    .join(
        batches.select(
            "batch_id",
            "supplier_id",
            "manufacturing_date",
            "expiry_date",
            "batch_quantity",
            "batch_status",
            "quality_status"
        ),
        on="batch_id",
        how="left"
    )
)

In [0]:
df_quality_gold = (
    df_quality_gold
    .join(
        drugs.select(
            "drug_id",
            "drug_name",
            "generic_name",
            "brand_name",
            "drug_category"
        ),
        on="drug_id",
        how="left"
    )
)

In [0]:
from pyspark.sql.functions import col, when

In [0]:
df_quality_gold = df_quality_gold.withColumn(
    "risk_category",
    when(
        col("severity") == "Critical",
        "High Risk"
    )
    .when(
        col("severity") == "High",
        "High Risk"
    )
    .when(
        col("severity") == "Medium",
        "Medium Risk"
    )
    .otherwise(
        "Low Risk"
    )
)

In [0]:
df_quality_gold = df_quality_gold.withColumn(
    "resolution_category",
    when(
        col("event_status").isin("Resolved", "Closed"),
        "Resolved"
    )
    .otherwise(
        "Open"
    )
)

In [0]:
df_quality_gold = df_quality_gold.select(
    "quality_event_id",
    "batch_id",
    "drug_id",
    "drug_name",
    "generic_name",
    "brand_name",
    "drug_category",
    "supplier_id",
    "event_date",
    "event_type",
    "severity",
    "risk_category",
    "event_status",
    "resolution_category",
    "description",
    "manufacturing_date",
    "expiry_date",
    "batch_quantity",
    "batch_status",
    "quality_status"
)

In [0]:
display(df_quality_gold)


In [0]:
df_quality_gold.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(
        "workspace.pharma_gold.gold_quality_analytics"
    )